In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from torch.cuda.amp import autocast, GradScaler
from tqdm.auto import tqdm
import os
import math
import csv

from model_encoder_decoder import Transformer

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_PATH = "../data/train_data.tsv"
BATCH_SIZE = 128
LR = 0.0002
EPOCHS = 20
D_MODEL = 256
N_LAYERS = 4
MAX_LEN = 1000
VOCAB_SIZE = 30000

CHECKPOINT_PATH = "checkpoint_encoder_decoder.pt"
FINAL_MODEL_PATH = "weights_encoder_decoder.pt"
METRICS_FILE = "metrics_encoder_decoder.csv"
RESUME = False

def train_tokenizer_and_process():
    if not os.path.exists("tokenizer_encoder_decoder.json"):
        print("Training Tokenizer...")
        tokenizer = Tokenizer(BPE(unk_token="<unk>"))
        tokenizer.pre_tokenizer = Whitespace()
        trainer = BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=["<pad>", "<sos>", "<eos>", "<unk>"], min_frequency=2)
        
        def data_gen():
            with open(DATA_PATH, "r", encoding="utf-8") as f:
                for line in f:
                    parts = line.strip().split('\t')
                    if len(parts) >= 4:
                        yield parts[1] # Eng
                        yield parts[3] # Rus

        tokenizer.train_from_iterator(data_gen(), trainer)
        tokenizer.save("tokenizer_encoder_decoder.json")
    else:
        print("Loading Tokenizer...")
        tokenizer = Tokenizer.from_file("tokenizer_encoder_decoder.json")

    if not os.path.exists("dataset_tensors_encoder_decoder.pt"):
        print("Tokenizing entire dataset...")
        src_seqs = []
        tgt_seqs = []
        
        sos_id = tokenizer.token_to_id("<sos>")
        eos_id = tokenizer.token_to_id("<eos>")

        with open(DATA_PATH, "r", encoding="utf-8") as f:
            for i, line in enumerate(f):
                if i % 100000 == 0: print(f"Processed {i} lines")
                parts = line.strip().split('\t')
                if len(parts) >= 4:
                    src_encoded = tokenizer.encode(parts[1]).ids
                    tgt_encoded = tokenizer.encode(parts[3]).ids
                    
                    src_seqs.append(torch.tensor([sos_id] + src_encoded + [eos_id], dtype=torch.int16))
                    tgt_seqs.append(torch.tensor([sos_id] + tgt_encoded + [eos_id], dtype=torch.int16))

        print("Saving tensors to disk...")
        torch.save({"src": src_seqs, "tgt": tgt_seqs}, "dataset_tensors_encoder_decoder.pt")
    else:
        print("Dataset tensors already exist.")

class CachedDataset(Dataset):
    def __init__(self, tensors_path):
        print("Loading tensors into RAM...")
        data = torch.load(tensors_path)
        self.src_list = data["src"]
        self.tgt_list = data["tgt"]
        print(f"Loaded {len(self.src_list)} examples.")

    def __len__(self):
        return len(self.src_list)

    def __getitem__(self, idx):
        return self.src_list[idx].long(), self.tgt_list[idx].long()

def collate_batch(batch, pad_idx):
    src_list, tgt_list = zip(*batch)
    
    src_list = [s[:MAX_LEN] for s in src_list]
    tgt_list = [t[:MAX_LEN] for t in tgt_list]
    
    src_padded = torch.nn.utils.rnn.pad_sequence(src_list, padding_value=pad_idx, batch_first=True)
    tgt_padded = torch.nn.utils.rnn.pad_sequence(tgt_list, padding_value=pad_idx, batch_first=True)
    return src_padded, tgt_padded

def calc_accuracy(logits, targets, pad_idx):
    preds = logits.argmax(dim=-1)
    mask = (targets != pad_idx)
    correct = (preds == targets) & mask
    return correct.sum().float(), mask.sum().float()

if __name__ == "__main__":
    train_tokenizer_and_process()
    
    tokenizer = Tokenizer.from_file("tokenizer_encoder_decoder.json")
    PAD_IDX = tokenizer.token_to_id("<pad>")
    VOCAB_SIZE = tokenizer.get_vocab_size()

    dataset = CachedDataset("dataset_tensors_encoder_decoder.pt")
    train_size = int(0.95 * len(dataset))
    val_size = len(dataset) - train_size
    train_data, val_data = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, 
                              collate_fn=lambda x: collate_batch(x, PAD_IDX), num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, 
                            collate_fn=lambda x: collate_batch(x, PAD_IDX), num_workers=0)

    model = Transformer(
        vocab_size_seq=VOCAB_SIZE,
        vocab_size_target=VOCAB_SIZE,
        d_model=D_MODEL,
        n_layer=N_LAYERS,
        n_head=8,
        d_head=D_MODEL // 8,
        d_ff=D_MODEL * 4,
        max_len=MAX_LEN,
        dropout=0.1,
        pad_idx=PAD_IDX
    ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR, betas=(0.9, 0.98), eps=1e-9)
    scaler = GradScaler()
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
    start_epoch = 0

    if RESUME and os.path.exists(CHECKPOINT_PATH):
        print("Resuming from checkpoint...")
        checkpoint = torch.load(CHECKPOINT_PATH)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch'] + 1

    if not RESUME or not os.path.exists(METRICS_FILE):
        with open(METRICS_FILE, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['Epoch', 'Train Loss', 'Train PPL', 'Train Acc', 'Val Loss', 'Val PPL', 'Val Acc'])

    print(f"Starting training on {DEVICE}...")
    
    for epoch in range(start_epoch, EPOCHS):
        model.train()
        train_loss_sum = 0
        train_correct_sum = 0
        train_total_tokens = 0
        
        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
        
        for src, tgt in loop:
            src, tgt = src.to(DEVICE), tgt.to(DEVICE)
            tgt_input = tgt[:, :-1]
            tgt_y = tgt[:, 1:]
            
            optimizer.zero_grad()
            
            with autocast():
                logits = model(src, tgt_input)
                loss = criterion(logits.reshape(-1, VOCAB_SIZE), tgt_y.reshape(-1))
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            
            loss_val = loss.item()
            train_loss_sum += loss_val
            
            with torch.no_grad():
                correct, total = calc_accuracy(logits, tgt_y, PAD_IDX)
                train_correct_sum += correct
                train_total_tokens += total
            
            loop.set_postfix(loss=loss_val)

        avg_train_loss = train_loss_sum / len(train_loader)
        avg_train_acc = (train_correct_sum / train_total_tokens).item()
        train_ppl = math.exp(avg_train_loss) if avg_train_loss < 100 else float('inf')

        model.eval()
        val_loss_sum = 0
        val_correct_sum = 0
        val_total_tokens = 0
        
        with torch.no_grad():
            for src, tgt in tqdm(val_loader, desc="Validation"):
                src, tgt = src.to(DEVICE), tgt.to(DEVICE)
                tgt_input = tgt[:, :-1]
                tgt_y = tgt[:, 1:]
                
                with autocast():
                    logits = model(src, tgt_input)
                    loss = criterion(logits.reshape(-1, VOCAB_SIZE), tgt_y.reshape(-1))
                
                val_loss_sum += loss.item()
                correct, total = calc_accuracy(logits, tgt_y, PAD_IDX)
                val_correct_sum += correct
                val_total_tokens += total

        avg_val_loss = val_loss_sum / len(val_loader)
        avg_val_acc = (val_correct_sum / val_total_tokens).item()
        val_ppl = math.exp(avg_val_loss) if avg_val_loss < 100 else float('inf')

        print(f"Ep {epoch+1} | Train Loss: {avg_train_loss:.4f} PPL: {train_ppl:.2f} Acc: {avg_train_acc:.4f} | "
              f"Val Loss: {avg_val_loss:.4f} PPL: {val_ppl:.2f} Acc: {avg_val_acc:.4f}")

        with open(METRICS_FILE, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([epoch+1, f"{avg_train_loss:.4f}", f"{train_ppl:.4f}", f"{avg_train_acc:.4f}", f"{avg_val_loss:.4f}", f"{val_ppl:.4f}", f"{avg_val_acc:.4f}"])

        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_train_loss,
        }, CHECKPOINT_PATH)

    # --- ФИНАЛЬНОЕ СОХРАНЕНИЕ ВЕСОВ ---
    print("Saving final model...")
    torch.save(model.state_dict(), FINAL_MODEL_PATH)
    print("Model Saved")
    print(f"Done. Metrics saved to {METRICS_FILE}")


Loading Tokenizer...
Dataset tensors already exist.
Loading tensors into RAM...
Loaded 760402 examples.


/tmp/ipykernel_3879/1078953892.py:151: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Starting training on cuda...


Epoch 1/20 [Train]:   0%|          | 0/5644 [00:00<?, ?it/s]/tmp/ipykernel_3879/1078953892.py:189: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Validation: 100%|██████████| 298/298 [00:04<00:00, 70.66it/s]


Ep 15 | Train Loss: 1.3556 PPL: 3.88 Acc: 0.7383 | Val Loss: 1.2741 PPL: 3.58 Acc: 0.7535


Epoch 16/20 [Train]:   6%|▌         | 350/5644 [00:14<03:35, 24.53it/s, loss=1.17] 